In [1]:
# P1-fix: os caminhos deste notebook sao RELATIVOS a EXIST_VIDEOS/ (plano 11 §3:
# "kernel do Jupyter em /home/a67606/Thesis/EXIST_VIDEOS ou cd ate la"). Se o kernel
# nascer noutro cwd, o assert do JSON da celula seguinte falha. Idempotente.
import os
from pathlib import Path

_TARGET = Path("/home/a67606/Thesis/EXIST_VIDEOS")
if Path.cwd() != _TARGET:
    assert _TARGET.is_dir(), f"diretorio do projeto nao encontrado: {_TARGET}"
    os.chdir(_TARGET)
assert Path("Dataset").is_dir(), f"cwd errado: {Path.cwd()} (sem Dataset/)"
print("cwd:", Path.cwd())


cwd: /home/a67606/Thesis/EXIST_VIDEOS


In [2]:
# Diagnóstico do residual: 9 itens com ffprobe>30 s que geraram 1 janela só.
# Replica load_audio_mono_16k EXATAMENTE (mono-mix → resample → ceil(n/480000)),
# porque um desvio causado pelo arredondamento do resampler é um achado DIFERENTE
# de um causado por pacotes truncados no decode.
import csv, math
from pathlib import Path
import pandas as pd, torch, torchaudio
import torchaudio.transforms as T
from tqdm import tqdm

TARGET_SR, CHUNK = 16000, 30 * 16000
df = pd.read_json("Dataset/training/EXIST2026_training.json").transpose()
name_by_id = {str(r["id_EXIST"]): Path(str(r["path_video"])).name for _, r in df.iterrows()}
rows = [r for r in csv.DictReader(open("analysis/audio_durations.csv"))
        if r["split"] == "training"]

cache = {}
def decoded_seconds(path):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        if sr not in cache:
            cache[sr] = T.Resample(orig_freq=sr, new_freq=TARGET_SR)
        wav = cache[sr](wav)
    return wav.shape[-1]

hist, culpados = {}, []
for r in tqdm(rows, desc="decodificando os 2524"):
    iid, dur = str(r["id_EXIST"]), float(r["duration_s"])
    n = decoded_seconds(f"Dataset/training/videos/{name_by_id[iid]}")
    w = max(1, math.ceil(n / CHUNK))
    hist[w] = hist.get(w, 0) + 1
    if dur > 30.0 and w == 1:
        culpados.append((iid, dur, n / TARGET_SR, dur - n / TARGET_SR))

print("\nhistograma recontado:", dict(sorted(hist.items())))
print("multi recontado:", sum(v for k, v in hist.items() if k > 1),
    " (o script gravou 835; se a recontagem der 842, o problema está na RUN, "
    "não no decode — investigação totalmente diferente)")
print(f"\nitens com ffprobe>30 s mas 1 janela: {len(culpados)} (esperado 9)")
print(f"{'id':>10} {'ffprobe':>9} {'decodif.':>9} {'perdido':>9}")
for iid, dur, dec, delta in sorted(culpados, key=lambda x: -x[3]):
    print(f"{iid:>10} {dur:9.3f} {dec:9.3f} {delta:9.3f}")

decodificando os 2524: 100%|█████████████████████████████████████████████████████████| 2524/2524 [02:01<00:00, 20.74it/s]


histograma recontado: {1: 1689, 2: 813, 3: 22}
multi recontado: 835  (o script gravou 835; se a recontagem der 842, o problema está na RUN, não no decode — investigação totalmente diferente)

itens com ffprobe>30 s mas 1 janela: 9 (esperado 9)
        id   ffprobe  decodif.   perdido
    220471    59.070     1.232    57.838
    220203    58.164     6.248    51.916
    220516    56.655     5.040    51.615
    220589    42.351     1.557    40.794
    220390    41.771     5.179    36.591
    220283    59.047    23.709    35.338
    220966    36.175     1.929    34.246
    220073    34.434     0.789    33.644
    220541    37.034    10.288    26.746


In [3]:
import whisper, torchaudio, pandas as pd
from pathlib import Path
IDS = ["220471","220203","220516","220589","220390","220283","220966","220073","220541"]
df = pd.read_json("Dataset/training/EXIST2026_training.json").transpose()
name_by_id = {str(r["id_EXIST"]): Path(str(r["path_video"])).name for _, r in df.iterrows()}
for iid in IDS:
    p = f"Dataset/training/videos/{name_by_id[iid]}"
    a = whisper.load_audio(p)                    # ffmpeg subprocess, 16k mono
    w, sr = torchaudio.load(p)
    print(f"{iid}: ffmpeg={len(a)/16000:8.3f}s  torchaudio={w.shape[-1]/sr:8.3f}s  sr={sr}")

220471: ffmpeg=   1.232s  torchaudio=   1.232s  sr=44100
220203: ffmpeg=   6.248s  torchaudio=   6.248s  sr=44100
220516: ffmpeg=   5.040s  torchaudio=   5.040s  sr=44100
220589: ffmpeg=   1.557s  torchaudio=   1.557s  sr=44100
220390: ffmpeg=   5.179s  torchaudio=   5.179s  sr=44100
220283: ffmpeg=  23.709s  torchaudio=  23.709s  sr=44100
220966: ffmpeg=   1.929s  torchaudio=   1.929s  sr=44100
220073: ffmpeg=   0.789s  torchaudio=   0.789s  sr=44100
220541: ffmpeg=  10.288s  torchaudio=  10.288s  sr=44100


In [5]:
# Lê ~31 GB do disco LOCAL da VM (nunca pelo mount). ~5–15 min.
import torch
from pathlib import Path

EXPECT_SHAPE = (64, 3, 224, 224)
all_ok = True
for split in ["training", "test"]:
    d = Path("frames/v2") / split
    bad_zero, bad_meta = [], []
    for p in sorted(d.glob("*.pt")):
        t = torch.load(p, weights_only=True)
        if t.dtype != torch.uint8 or tuple(t.shape) != EXPECT_SHAPE:
            bad_meta.append((p.name, str(t.dtype), tuple(t.shape)))
        elif not t.any():
            bad_zero.append(p.name)
    print(f"{split}: dtype/shape ruins={len(bad_meta)} todo-zeros={len(bad_zero)}")
    if bad_meta[:3]: print("   ex.:", bad_meta[:3])
    if bad_zero[:3]: print("   ex.:", bad_zero[:3])
    all_ok &= not bad_meta and not bad_zero
print("testes 5+8:", "PASS" if all_ok else "FAIL")
# Referência v1: 0/3198 todo-zeros (analysis/frames_zero_scan.json). Na v2 um
# todo-zeros é ainda mais impossível por construção (falha não grava arquivo).

# Áudio: mesma varredura, barata (26 MB)
for split in ["training", "test"]:
    d = Path("audio_features/v2") / split
    bad = [p.name for p in d.glob("*.pt")
            if not torch.load(p, weights_only=True).any()]
    print(f"áudio {split}: todo-zeros={len(bad)}", "ok" if not bad else "FAIL")

training: dtype/shape ruins=0 todo-zeros=0
test: dtype/shape ruins=0 todo-zeros=0
testes 5+8: PASS
áudio training: todo-zeros=0 ok
áudio test: todo-zeros=0 ok


In [6]:
import sys; sys.path.insert(0, ".")
import pandas as pd
import torch
import video_core as vc
from sklearn.preprocessing import RobustScaler
from transformers import XLMRobertaTokenizer

full_df = pd.read_json("Dataset/training/EXIST2026_training.json").transpose()
train_df, val_df = vc.build_frozen_split(full_df)      # carrega o artefato congelado
# scaler ajustado SÓ no treino (anti-leakage) — mesmo procedimento do notebook de treino
scaler = RobustScaler().fit(vc.collect_bio_vectors(train_df))
tokenizer = XLMRobertaTokenizer.from_pretrained(vc.XLMR_NAME)
ds = vc.EXISTVideoDataset(train_df, tokenizer, scaler,
                        vc.frames_dir("train"), vc.audio_dir("train"))
item = ds[0]
assert item["frames"].shape == (16, 3, 224, 224) and item["frames"].dtype == torch.float32
assert item["audio"].shape == (1280,)
# a faixa da v1 medida era [−2,10; +2,64] (OPT-01) — a v2 tem de cair na mesma:
lo, hi = item["frames"].min().item(), item["frames"].max().item()
assert -2.5 < lo < 0 and 0 < hi < 3.0, (lo, hi)
print("smoke test do consumidor: PASS", (lo, hi))

[split] carregado de Dataset/val_split_42.json (seed=42, estratificação=task_a): 2019 treino / 505 val
[bio] 4038 linhas utilizador-vídeo | reaction_time imputado em 77 (1.91 %)


smoke test do consumidor: PASS (-2.1179039478302, 2.640000104904175)
